In [ ]:
#General Dependencies
from pathlib import Path
import glob, re
import pandas as pd
import numpy as np
from natsort import natsorted

#RDKit Dependencies
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import BondType
from rdkit.Chem import rdMolTransforms
from rdkit.Chem import GetPeriodicTable
from rdkit.Chem import PeriodicTable

"""
### Notebook Overview: 'Extract_Descriptors_from_DFT.ipynb'
Written by @GCH v1.0 - last updt. 05/06/2026

This notebook contains code to extract DFT-derived quantitative and qualitative descriptors from validated Orca Aryne.out files.
The extracted/calculated descriptors are written incrementally to a Pandas DataFrame called "working_df" before being written
to the local directory as "Aryne_Calculated_Molecular_Descriptors.csv". Two additional csv files are also written to the local
directory: "6_Membered_Calculated_Molecular_Descriptors.csv" and "6_Membered_Calculated_Molecular_Descriptors.csv" containing 
only 6- or only 5-membered Arynes, respectively. 

### Planned Features:
1. Ensure all methods are fully commented/have complete documentation
2. Better Error-Handling/exceptions
3. Add more/better descriptors

### Quantitative/Qualitative Molecular Descriptors Calculated by this Notebook:
1. Size of Aryne Ring - what is the number of atoms comprising the ring that the aryne C#C is in?
2. Fused or Monocyclic Aryne - Is the aryne in a monocyclic or multicyclic (fused) system?
3. Aryne Atom Indices - the 0-indexed indices of atoms comprising the C#C aryne bond.
4. Aryne Bond Atom Types - Both are C#C by definition but it's useful as a cell for future/other projects.
5. Aryne Bond Distance - the distance, in Angstroms, of the aryne C#C bond.
6. Aryne Bond Angle Indices - the 0-indexed atom indices of the C#C-X angle(s).
7. Aryne Neighbor Atom Types - the atom types of the aryne bond angles, ex: C#C-X => [C, C, X]
8. Aryne Bond Angles - the internal bond angles around the aryne bond, ex: C#C-X, in degrees.
9. Max Aryne Bond Angle - the larger of the two calculated bond angles comprising the aryne bond, in degrees.
10. Max Deviation of Bond Angle - The largest angle deviation from an equilateral polygon's ideal angles
11. Sum of Abs. Deviation - the sum of both bond angles' absolute deviation from an ideal polygon
12. Aryne Dihedral Angle - the dihedral angle, in degrees, of the X-C#C-Y dihedral angle.
13. HOMO/LUMO Energies - in eV or Hartrees, the DFT-energies of the FMOs.
14. Aryne Hirshfeld Charges - the Hirshfeld charge of each aryne C atom.
15. Sum of Abs Hirshfeld Charge - the absolute charge summed across both C#C carbon atoms. 

### How to Use this Notebook:
1. Define relevant Paths to necessary files in the PATH cell, below.
2. Define relevant output Paths for new files to be written to in the PATH cell, below.
3. Run all cells in the notebook to generate molecular descriptors and add them to a dataframe
4. An output .csv containing the calculated molecular descriptor data is written to the local dir
5. Output .csvs are separately generated for 5- and 6-membered Arynes and written to the local dir

### Example:
I just used the previous module to calculate dehydration energies for all of my arenes and arynes. I now pass the
relevant directories containing aryne.outs, xyzs, and sdfs in the cell, below. Running each of the following cells
will populate a "working_df" Pandas DataFrame with calculated descriptors extracted from various files. At the end
of this process, several output .csvs containing extracted parameters are written to the local directory. For more
details, see the specific inputs/outputs, below. 

### Program Inputs/Outputs:
#### Input: 
1. Directory containing Orca 5.0.3 DFT output files (opt+freq; M06-2X-D3 / def2-SVP)
2. Directories for .xyzs and .sdfs extracted from the above .out files (made with previous Notebook)
3. "Calculated_Dehydrogenation_Energies.csv" - a .csv containing arenes/arynes and DFT-energies

#### Output:
1. "Aryne_Calculated_Molecular_Descriptors.csv" - all calculated descriptors and dehydration energies
2. "5_membered_Calculated_Molecular_Descriptors.csv" - Descriptors for 5-memebered hetarynes, only
3. "6_membered_Calculated_Molecular_Descriptors.csv" - Descriptors for 6-memebered hetarynes, only
"""

In [ ]:
""" ### Define PATHS/relevant directories / Import Data from 'calculated_dehydrogenation_energies.csv' ### 

This cell is where you will define Paths that will contain input/output files for the project. It is also
used to import the "Calculated_Dehydrogenation_Energies.csv" file from the preceding Notebook. 
"""

#top level project dir
project_dir = Path.cwd().resolve().parent

#Directory for Module6 where the dehydration energy df exists
mod6_dir = project_dir / "Module6_Extract_Dehydration_Energies"

#dir containing validated orca.out files
#Orca 5.0.3; LOT: M06-2X-D3 / def2-SVP 
aryne_dft_out_path = project_dir/"DFT_Aryne_Data"/"Aryne_Orca_DFT_Outputs"

#dir containing validated .sdfs written with DFT-optimized geometries
aryne_opt_sdf_path = aryne_dft_out_path/"Aryne_Post_Opt_sdfs"

#dir containing formatted .xyz files extracted from DFT .out files
aryne_opt_xyz_path = aryne_dft_out_path/"Aryne_Opt_xyz_Files"

#Read in the Arene/Aryne 'Calculated_Dehydrogenation_Energies.csv' dataset
dehyd_csv_name = mod6_dir / "Calculated_Dehydrogenation_Energies.csv"
incoming_data = pd.read_csv(dehyd_csv_name)

#Print to console
incoming_data.head()

In [ ]:
def get_filepaths_in_target_dir(target_directory: Path, extension: str, printing=True):
    """
    Function: returns a list of Path objects pertaining to files in a specified directory. Searches for specific .ext
    
    Input:
        - target_directory; Path containing files with the '.extension'
        - extension; str: a string specifying an extension to use, ex: ".out", ".inp", ".sdf" etc.
        - printing; bool; Default value = True; controls printing back to user
        
    Returns: 
        - filepaths: list; a list of filepaths for each file wit .extension' in 'target_directory'
    """
    
    #grab user extension with a search wildcard
    file_extension = f"*{extension}"
    
    #gather and sort .ext files in /target_directory/
    filepaths = target_directory.glob(file_extension)
    filepaths = natsorted(filepaths, key=str)

    #return some user info
    if printing:
        print(f"\033[1mFound {len(filepaths)} {extension} files in '/{target_directory.stem}/\033[0m'")

    #return the list of located filepaths
    return filepaths

In [ ]:
def sdf_to_mol_rdkit(path_to_sd_file: Path) -> Chem.Mol:
    """
    Function: Converts an SD-file containing a single molecule to an RDKit mol object. Returns the mol object. 
    Input:
        - path_to_sd_file: Path; a path to a single named .sd-file
    Returns:
        - mol: an RDKit MOL object (assuming a single mol in each .sdf)
    """
    #SDMolSupplier is a class that supplies mol data from an SD-file. SDFs can contain many mols in seq
    #so you need to "supply" them one at a time from the sdf. 
    supplier = Chem.SDMolSupplier(path_to_sd_file)

    mol = None # assign a default none to mol 
    
    for m in supplier: #for each of the objects pulled from the sdf and retained in "supplier"
        if m is not None: #If RDKit can interpret the object as a mol, 
            mol = m #assign the mol object to "mol"

        else: #if RDkit can't interpret the supplied object,
            continue #go to the next object in supplier; N/A here

    #return the mol if it's good
    return mol 

In [ ]:
def ring_size_of_aryne(path_to_sd_files: Path):
    """
    Function: Given a directory containing many .sdfs, determine the size of the ring that
        contains the aryne bond in each .sdf

    Input:
        - path_to_sdf_files: Path; A path to a directory containing optimized/validated .sdfs

    Returns:
        - aryne_ring_size_list: list; A list of integer values representing the size of the ring
            (number of atoms making up the ring) that contains the aryne C#C Bond. 
    """
    
    #get the list of .sdf files in /path_to_sdf_files/ to extract from
    target_sd_files = get_filepaths_in_target_dir(path_to_sd_files, ".sdf", printing=False)

    #grab the bond indices for each aryne
    aryne_bond_indices = []
    
    #for each .sdf file
    for sd_file in target_sd_files:
        bond_indices = get_aryne_bond_indices(sd_file)
        aryne_bond_indices.append(bond_indices)
        
    print(f'Num bond_indices: {len(aryne_bond_indices)}')

    #loop over the .sdf files and find the ring size of the ring the aryne bond is in
    aryne_ring_size_list = []
    for sd_file, bond_index in zip(target_sd_files, aryne_bond_indices):

        #sdf to mol
        mol = sdf_to_mol_rdkit(sd_file)

        #grab local bond indices of the aryne
        atom1, atom2 = bond_index

        #check to see that the bond exists
        aryne_bond = mol.GetBondBetweenAtoms(atom1, atom2)
        if aryne_bond is not None:
            
            #print(f'Bond exists between atom [{atom1}, {atom2}].')
            bond_idx = aryne_bond.GetIdx()
        else:
            #print(f'No bond exists between atoms [{atom1}, {atom2}].')
            continue

        #ring info
        ring_info = mol.GetRingInfo()
        
        bond_rings = ring_info.BondRings()
        for ring in bond_rings:
            if bond_idx in ring:
                #print(f'Aryne is in a {len(ring)}-membered ring')
                size_of_ring = len(ring)
                aryne_ring_size_list.append(size_of_ring)

    return aryne_ring_size_list

In [ ]:
def is_aryne_in_fused_ring(path_to_sd_files: Path):
    """
    Function: Given a directory containing optimized .sdfs, determine if the aryne bond is in
        a mono-ring or in a fused ring system (ex: naphthalene is fuzed, benzee is mono)

    Input:
        - path_to_sd_files: Path; A path to a directory containing optimized/validated .sdfs

    Returns:
        - aryne_ring_types: list; a list of strs defining ring types; either 'mono' or 'multi'
    """
    
    #get the list of .sdf files in /path_to_sdf_files/ to extract from
    target_sd_files = get_filepaths_in_target_dir(path_to_sd_files, ".sdf", printing=False)

    #loop over the .sdf files and find the ring size of the ring the aryne bond is in
    aryne_ring_types = []
    for sd_file in target_sd_files:
        #sdf to mol
        mol = sdf_to_mol_rdkit(sd_file)

        #ring info
        ring_info = mol.GetRingInfo()
        bonds_in_rings = ring_info.BondRings()

        #return type depending on number of rings in the mol
        if len(bonds_in_rings) == 1: 
            ring_type = 'mono'
            aryne_ring_types.append(ring_type)
            
        elif len(bonds_in_rings) > 1:
            ring_type = 'multi'
            aryne_ring_types.append(ring_type)

    return aryne_ring_types

In [ ]:
def get_aryne_bond_indices(path_to_sd_file: Path):
    """
    Function: extract the 0-indexed atom indices of atoms comprising the C#C triple bond in arynes
    
    Input:
        - path_to_sd_file: Path; path to a validated post-optimization .SDF
        
    Returns:
        - aryne_bonds_in_mol; a pair of bond indices corresponding to a triple bond, if found: [x, y]
    """

    #convert all the .sdf files in path_to_sdfs to RDKit .mol objects
    aryne_mol = sdf_to_mol_rdkit(path_to_sd_file)
        
    #list to store atom indices that comprise the C#C triple bond
    aryne_bonds_in_mol = []
    
    #loop over each bond in the mol and determine if it's consistent with a C#C triple bond
    for bond in aryne_mol.GetBonds():

        #if the bond is a C#C triple bond (as determined by RDKIT)
        if bond.GetBondType() == BondType.TRIPLE:
            
            #get the X#Y atom indices
            aryne_c1 = bond.GetBeginAtomIdx()
            aryne_c2 = bond.GetEndAtomIdx()

            #append both atom indices to the list of atom indices
            aryne_bonds_in_mol.append([int(aryne_c1),int(aryne_c2)])

    #count the number of putative arynes in the molecule (sanity check)
    num_arynes_found = len(aryne_bonds_in_mol)

    #If there is a single aryne, this is the best case
    if num_arynes_found == 1:
        
        #return the aryne bond indices (it's a tuple)
        return aryne_bonds_in_mol[0]
        
    elif num_arynes_found == 0:
        print("Error: No aryne bonds found in imported .xyz coordinates")
        
    else:
        print("Something terrible and unexpected has happened.")  

In [ ]:
def calculate_aryne_bond_distance(conformer, bond_index_pair):
    """
    Function: Given a conformer and a bond_index_pair corresponding to atom indices
        comprising the aryne C#C bond, calculate the bond distance between the two 
        atom indices in Angstroms. 
    
    Input:
        - conformer; RDKit conformer object read from .sdf
        - bond_index_pair; a pair of atom indices [x, y] corresponding to the C#C 
            triple bond of the aryne
    Returns:
        - aryne_distance: float; The distance, in Angstroms, between atoms x and y
    """   
    
    #passing in the triple bond index as a list: [x, y]
    atom_idx1, atom_idx2 = bond_index_pair

    #get the xyz coords of each supplied atomic index as an np array 
    coords1 = np.array(conformer.GetAtomPosition(atom_idx1))
    coords2 = np.array(conformer.GetAtomPosition(atom_idx2))

    #calculate the distance (in angstroms) between the two atoms
    aryne_distance = np.linalg.norm(coords2 - coords1)

    #can optionally print the details for debug
    #print(f"The distance between atom {atom_idx1} and atom {atom_idx2} is: {aryne_distance:.3f} Angstroms")

    #return the calculated distance as float
    return aryne_distance

In [ ]:
def calc_all_aryne_bond_distances(path_to_opt_sd_files: Path, printing=True):
    """
    Function: Calculates the bond distance of the aryne C#C bond in units of Angstroms. Uses
        the post-optimization .sdfs generated by the "validate_and_extract_from_DFT" notebook. 

    Input:
        - path_to_opt_sd_files: Path; a directory containing validated post-optimization SDFs
        - printing: Bool; Optionally turn on/off printing to user

    Returns:
        - calculated_aryne_bond_distances: List of floats; C#C bond distances in Angstroms
    """

    if printing:
        print(f"Searching for .sd-files in '/{path_to_opt_sd_files.name}/'...")
    
    #get all the .sdfs in a specified directory
    opt_sd_files = get_filepaths_in_target_dir(path_to_opt_sd_files, ".sdf", printing=False)

    if printing:
        print(f"\nNow parsing {len(opt_sd_files)} .sd-files for Aryne Bond Distances (Angstrom)...\n")

    #init an empty list to store the calculated bond distances (in float)
    calculated_aryne_bond_distances = []

    #loop over the sd-files in the targeted directory
    for sd_file in opt_sd_files:

        #extract the C#C atom indices from the current .sdf
        aryne_bond_indices = get_aryne_bond_indices(sd_file)

        #create a RDKIT mol object
        opt_mol = sdf_to_mol_rdkit(sd_file)
        #assign to a specific coord. set (only 1 in each, so 0 index)
        conformer = opt_mol.GetConformer(0)

        #calculate the bond distance between the ID'ed atom indices
        aryne_bond_distance = calculate_aryne_bond_distance(conformer, aryne_bond_indices)

        #round the value to 4dps (using 4 dps across the proj)
        rounded_distance = round(aryne_bond_distance, 4)

        #add the rounded val to the list of aryne bond distances
        calculated_aryne_bond_distances.append(rounded_distance)

        if printing:
            print(f"'{sd_file.name}:\t Parsed Bond Distance (Ang): {rounded_distance}'")

    return calculated_aryne_bond_distances

In [ ]:
def get_aryne_bond_angle_indices(path_to_sd_files: Path, list_of_atom_indices: list):
    """
    Function: Passed a list of SDFs and a list of aryne C#C bond indices, get the neighboring atoms
        indices with respect to the aryne bond distances. Ex: if I had an aryne with bond indices [1, 2]
        that had neighboring atom indices of 0 and 3, this method will return [0, 1, 2] and [1, 2, 3].  

    Input:
        -path_to_sd_files: Path; A path to a directory containing optimized/validated .sdfs
        -list_of_atom_indices: list; a list of aryne bond indices comprising the C#C bond
        
    Returns: bond_angle_indices: list; a list of 3-element lists each describing an internal bond angle
        like C#C-X, ex: [0, 1, 2] describes the bond angle between atoms 0, 1, and 2. 
    """
    
    #get the list of .sdf files in /path_to_sdf_files/ to extract from
    target_sd_files = get_filepaths_in_target_dir(path_to_sd_files, ".sdf", printing=False)

    #init an empty list to store our captured bond angle indices: [x, y, z] 
    bond_angle_indices = []

    #iterate through the target .sdf files for atom indices neighboring the supplied index #
    for sd_file, atom_index in zip(target_sd_files, list_of_atom_indices):

        #convert to mol
        mol = sdf_to_mol_rdkit(sd_file)

        #Target the aryne atom index
        atom_of_interest = mol.GetAtomWithIdx(atom_index)

        #want to return an ordered list of atom indices defining a bond angle
        angle_indices = []

        #Get neighboring atoms of supplied atom index                                             
        neighbor_atoms = atom_of_interest.GetNeighbors()
        
        #Get the indices of the other two atoms
        for neighbor_atom in neighbor_atoms:

            #append the neighboring atoms to a list of angle indices
            angle_indices.append(int(neighbor_atom.GetIdx()))

        #what is the atom index that 
        inserted_atom = int(atom_of_interest.GetIdx())
        
        #inset the original index at the fulcrum of the angle (middle)
        final_angle = angle_indices.copy()

        #insert the non-aryne index at position 1 of the trio
        final_angle.insert(1, inserted_atom)

        #append the bond angle indices to the list of indices
        bond_angle_indices.append(final_angle)

    #return the list of aryne bond angle atom indices
    return bond_angle_indices

In [ ]:
def calculate_bond_angles(path_to_sd_files: Path, list_of_bond_angle_indices: list):
    """
    Function: Given a directory of optimized .sdfs and a list of atom indices comprising aryne
    bond angles, calculate the internal bond angle of the trio of angle indices and add the 
    calculated value to a list of bond angles. Returns the list of calculated bond angles. 

    Input:
        -path_to_sdf_files: Path; A path to a directory containing optimized/validated .sdfs
        -list_of_bond_angle_indices: list; 

    Returns:
        - calculated_bond_angles: list; a list of calculated bond angles in degrees (floats)
    """
    
    #get the list of .sdf files in /path_to_sdf_files/ to extract from
    target_sd_files = get_filepaths_in_target_dir(path_to_sd_files, ".sdf", printing=False)

    #init an empty list to store the calculated bond angles in degrees (type: float)
    calculated_bond_angles = []
    
    for sd_file, bond_angle_set in zip(target_sd_files, list_of_bond_angle_indices):
        #pull in the .xyz coordinates and make a conf object
        optimized_mol = sdf_to_mol_rdkit(sd_file)
        conformer = optimized_mol.GetConformer(0)

        #pass atom_IDs comprising the bond
        a1_idx, a2_idx, a3_idx = bond_angle_set

        #calculate the internal bond angle (degrees)
        angle_deg = rdMolTransforms.GetAngleDeg(conformer, a1_idx, a2_idx, a3_idx)

        #round the calculated bond angle to 4 dps (used across the dataset)
        rounded_deg = round(angle_deg, 4)

        #add the calculated bond angle to the list of bond angles
        calculated_bond_angles.append(rounded_deg)

    #return the list of calculated bond angles
    return calculated_bond_angles

In [ ]:
def get_aryne_dihedral_indices(path_to_sdf: Path):
    """
    Function:
    
    Input:
        -path_to_sdf: Path; Filepath to an .sdf (presumes a single structure in each .sdf)
    
    Returns:
        dihedral_indices: list; list of atom indices corresponding to the dihedral angle 
            comprising a triple bond, if found: [i, x, y, j]
    """

    #convert sdf to mol
    mol = sdf_to_mol_rdkit(path_to_sdf)

    #stores atom indices determined to be triple
    aryne_bonds_in_mol = []
    #loop over each bond in the mol and determine if it's consistent with a X#X triple bond
    for bond in mol.GetBonds():
        if bond.GetBondType() == BondType.TRIPLE:
            #get the X#Y atom indices
            aryne_c1 = bond.GetBeginAtomIdx()
            aryne_c2 = bond.GetEndAtomIdx()
            aryne_bonds_in_mol.append([int(aryne_c1),int(aryne_c2)])

        num_arynes_found = len(aryne_bonds_in_mol)

    if num_arynes_found == 1:
        aryne_indices = aryne_bonds_in_mol[0]

        neighbor_indices = []
        #work with 1 side, first
        for aryne_index in aryne_indices:
            #center on a target aryne atom
            aryne_atom = mol.GetAtomWithIdx(aryne_index)
            neighbor_atoms = aryne_atom.GetNeighbors()

            #add only the non-aryne neighbor to the list of neighbors
            for neighbor_atom in neighbor_atoms:
                #rdkit is 0 indexed but a lot of Gui are 1-indexed
                if neighbor_atom.GetIdx() not in aryne_indices:
                    neighbor_indices.append(neighbor_atom.GetIdx())
                    #print(f'Appending new atom: {neighbor_atom.GetIdx()}')
                else:
                    #print(f'Atom already in the List: {neighbor_atom.GetIdx()}')
                    continue

        #build the dihedral
        atom_a = neighbor_indices[0]
        atom_b = aryne_indices[0]
        atom_c = aryne_indices[1]
        atom_d = neighbor_indices[1]
    
        dihedral_indices = [atom_a, atom_b, atom_c, atom_d]
        
        return dihedral_indices
                         
    ### Error things ### 
    elif num_arynes_found == 0:
        print("Error: No aryne bonds found in imported .xyz coordinates")
        
    else:
        print("Something terrible has happened")  

In [ ]:
def calculate_aryne_dihedral_degrees(conformer, dihedral_bond_indices: list):
    """
    Function:
    
    Input:
        - conformer
        - dihedral_bond_indices: list
        
    Returns:
        - The dihedral angle defined by [a, x, y, b] in degrees
    """   
    
    #passing in the triple bond index as a list: [x, y]
    atom_a, atom_b, atom_c, atom_d = dihedral_bond_indices

    dihedral_deg = rdMolTransforms.GetDihedralDeg(conformer, atom_a, atom_b, atom_c, atom_d)
    abs_dih = abs(round(dihedral_deg, 3))

    #can optionally print the details for debug
    #print(f"The absolute value of the dihedral angle between atoms {dihedral_bond_indices} is: {abs_dih:.3f} degrees")

    return abs_dih

In [ ]:
def calc_all_aryne_dihedral_angles(path_to_opt_sd_files: Path):
    """
    Function: Given a directory containing many .sdfs, 

    Input: 
        -path_to_opt_sd_files: Path; A path to a directory containing optimized/validated .sdfs

    Returns:
        - calculated_aryne_dihedral_angles: list; a list of calculated dihedral angles (degrees)
            corresponding to the dihedral angle of the aryne bond
    """

    #get the list of output files in 'target_directory' to extract from
    opt_sd_files = get_filepaths_in_target_dir(path_to_opt_sd_files, ".sdf", printing=False)

    #Init an empty list to store the calculated dihedral angle
    calculated_aryne_dihedral_angles = []
    
    #for each .sdf in the target dir,
    for sd_file in opt_sd_files:

        #Get the indices corresponding to the dihedral centered on the aryne
        aryne_dihedral_indices = get_aryne_dihedral_indices(sd_file)

        #covert the optimized .sdf to a mol object
        optimized_mol = sdf_to_mol_rdkit(sd_file)
        
        #assign it the first .xyz coords in the .sdf file
        conformer = optimized_mol.GetConformer(0)

        #calculate the aryne bond distance (in Angstroms)
        aryne_dihedral_angle = calculate_aryne_dihedral_degrees(conformer, aryne_dihedral_indices)
        
        #append it
        calculated_aryne_dihedral_angles.append(aryne_dihedral_angle)

    return calculated_aryne_dihedral_angles

In [ ]:
def calc_dev_from_ideal_angle(bond_angle_in_degrees: float, n_sided_polygon: int):
    """
    Function:

    Input:
        - bond_angle_in_degrees: float; The calculated bond angle of a ring angle in degrees
        - n_sided_polygon: int; the number of sides in a polygon, ex: 5 or 6 for pentagon/hexagon

    Returns:
        - deviation_from_ideal: float; how much a given bond angle deviates from its
            "ideal" bond angle as determined by the polygon internal angle formula; units = degrees.
    """

    #how many sides in the polygon
    num_sides = int(n_sided_polygon)

    #calculate the "ideal" internal angle for an equilateral n-sided polygon; [(n-2)*180]/n
    ideal_angle = (((num_sides - 2) * 180) / num_sides)

    #calculate how much a given angle deviates from its "ideal" angle (in degrees)
    deviation_from_ideal = round((bond_angle_in_degrees - ideal_angle), 2)

    #return the deviation
    return deviation_from_ideal

In [ ]:
def get_orbital_data_orca(orca_dft_output_filepath: Path):
    """
    Function: given a validated DFT output file (Orca 5.0.3), extract the orbital data from
        the output file. 

    Input:
        - orca_dft_output_filepath: Path; a filepath corresponding to a validated Orca output file

    Returns:
        - orbital_df: Pandas DF; a dataframe containing the formatted orbital data from the .out file
    """
    
    #check to see that the file exists
    if orca_dft_output_filepath.is_file():

        #set up to extract blocks of text (multiple orbital blocksin an .out, potentially)
        extracted_blocks = []
        current_block_lines = []
        printing = False

        #seems like orca jobs for some reason can have orbital blocks ending in 
        #"* MULLIKEN POPULATION ANALYSIS *" or "MOLECULAR ORBITALS" - one has a 
        #blank line and one doesnt so we need to explicitly read them correctly
        #to ensure the whole block is captured; these keep track of which case. 
        mulliken = False
        mol_orbs = False

        try:
            #read through the output file collecting lines of text defining 'orbital blocks'
            #aka raw orbital data
            with open(orca_dft_output_filepath, "r") as file:
                for line in file:
                    stripped_line = line.lstrip()
                    
                    #Read through lines until hitting the Final Geometry Block indicator in text
                    if stripped_line.startswith("NO   OCC          E(Eh)            E(eV)"):
                        printing = True
                        #reset the lines captured
                        current_block_lines = []
                        continue
    
                    #printing is on; now look for the end of the block to stop capturing
                    if printing:
                        #block can end with: ------ + MOLECULAR ORBITALS underneath;
                        if stripped_line.startswith("-"):
                            #have reached the end of the orbital block
                            printing = False
    
                            #raise the flag for the case matched
                            mol_orbs = True
                            
                            #append the captured lines to the blocks
                            extracted_blocks.append(current_block_lines)
                            continue
    
                        #block can end with ****** + * MULLIKEN POPULATION ANALYSIS; 
                        elif stripped_line.startswith("*"):
                            #have reached the end of the orbital block
                            printing = False
    
                            #raise the flag for the case matched
                            mulliken = True
    
                            #append the captured lines to the blocks
                            extracted_blocks.append(current_block_lines)
                            continue
                            
                        else:
                            #print(stripped_line)
                            current_block_lines.append(line.strip())
                            
        except Exception as e:
                print(f"Could not read .out {orca_dft_output_filepath.name}: {e}")

        if mol_orbs:
            #we care about the optimized orbital energies, so get the last one
            final_orbital_energies = extracted_blocks[-1]
            #there's no trailing blank line in this case

        if mulliken:
            #we care about the optimized orbital energies, so get the last one
            last_block = extracted_blocks[-1]
            
            #there's a trailing line of blank text; remove that
            final_orbital_energies = last_block[:-1]

        #loop over the raw text and set up to convert to DF 
        orbital_format = []
        for line in final_orbital_energies:
            stripped = line.strip()
            orbital_line = stripped.split()
            orbital_format.append(orbital_line)

        #generate a df from the formatted text
        orbital_df = pd.DataFrame(orbital_format, columns=["OrbNum", "Occup.", "E(Eh)", "E(eV)"])
        
        orbital_df['Occup.'] = pd.to_numeric(orbital_df['Occup.'])
        #orbital_df = orbital_df['Occup.'].astype(int)

        #return the orbital data as a Pandas DF
        return orbital_df

In [ ]:
def get_homo_lumo_energies_orca(dataframe: pd.DataFrame, preferred_units: str):
    """
    Function: Extract the HOMO/LUMO orbital energies from a dataframe containing orbital data
        extracted from DFT .out files (Orca 5.0.3)

    Input:
        - dataframe: pd.DataFrame; 
        - preferred_units: str; either "eV" or "hartree"
        
    Returns:
        - homo_energy: float; 
        - lumo_energy: float; 
    """
    
    #split the incoming df into occupied/virtual orbital frames
    #Will extract HOMO energy from last row of this DF:
    occupied_df = dataframe[dataframe['Occup.'] == 2.0]

    #Will extract LUMO energy from first row of this DF:
    virtual_df = dataframe[dataframe['Occup.'] == 0.0]

    #if the units = 'eV', return in electron volts
    if preferred_units == 'eV':
        homo_energy = occupied_df.tail(1).iat[0 ,3]
        lumo_energy = virtual_df.head(1).iat[0 ,3]

    #if the units = 'Hartree', return in Hartree
    elif preferred_units == 'hartree':
        homo_energy = occupied_df.tail(1).iat[0 ,2]
        lumo_energy = virtual_df.head(1).iat[0 ,2]

    #return FMO energies in desired units
    return homo_energy, lumo_energy

In [ ]:
def extract_homo_lumo_energy_data(path_to_dft_out_files: Path, printing=True):
    """
    Function: Given a directory of validated DFT .out files (Orca 5.0.3), Extract the HOMO/LUMO
        energies in either eV or Hatrees from each of the .out files. Returns lists of HOMO and
        LUMO energies. 

    Input:
        -path_to_dft_out_files: Path; path to a directory containing validated DFT
            .out files (Orca 5.0.3)
        -printing: Bool; optionally turn on/off printing to user for transparency

    Returns:
        - homo_energies: list; a list of extracted HOMO energies
        - lumo_energies: list; a list of extracted LUMO energies
    """
    
    #get the list of .out files in /path_to_dft_out_files/
    target_out_files = get_filepaths_in_target_dir(path_to_dft_out_files, ".out", printing=False)

    #optionally print updates to the user
    if printing:
        print(f'\nProcessing {len(target_out_files)} Orca.outs in {path_to_dft_out_files}...')

    #Init some empty lists to store our extracted data
    homo_energies = []
    lumo_energies = []

    #loop over the .out files in the target dir.
    for out_file in target_out_files: 

        print(f"Working .out: {out_file.name}")

        #use the get_orbital_data_orca() method to get the raw orbital data from the current .out
        orbital_df = get_orbital_data_orca(out_file)

        #with a DF in-hand, extract the HOMO and LUMO energies, here using units of eV
        homo_energy, lumo_energy = get_homo_lumo_energies_orca(orbital_df, 'eV')

        #append those FMO energies to the initialized lists for returning
        homo_energies.append(homo_energy)
        lumo_energies.append(lumo_energy)

    if printing:
        print(f'\nFinished extracting from {len(target_out_files)} Orca.outs in {path_to_dft_out_files.name}...')

    #return the collected data
    return homo_energies, lumo_energies

In [ ]:
def get_hirshfeld_textblock_orca(orca_dft_output_filepath: Path):
    """
    Function: Given a validated Orca .out file (Orca 5.0.3) containing Hirshfeld charge data, extract the
        Hirshfeld charge data as a Pandas DF for parsing with the 'get_aryne_hirshfeld_charges()' method.

    Input:
        - orca_dft_output_filepath: Path; a filepath to a validated Orca DFT .out file (Orca 5.0.3)

    Returns:
        - hirshfeld_df: pd.Dataframe; a formatted Pandas DF containing Hirshfeld charge data
    """
    
    if orca_dft_output_filepath.is_file():
        #set up to extract blocks of text (multiple orbital blocksin an .out, potentially)
        extracted_blocks = []
        printing = False #start with printing "off"

        #Parse through output file for Hirshfeld Charge section(s)
        try:
            with open(orca_dft_output_filepath, "r") as file:                
                #read large files line-by-line
                for line in file:
                    stripped_line = line.lstrip() #remove leading whitespace
                    
                    #Read through lines until finding the start of a Hirshfeld charge section
                    if stripped_line.startswith("HIRSHFELD ANALYSIS"):
                        current_hirsh_block = [] #reset the current block since it's start of new section
                        printing = True #start capturing lines
                        continue #continue the loop

                    if printing: #we have found the start of a hirsh block

                        #if we reach the end of the block,
                        if stripped_line.startswith("TOTAL"):
                            #turn off line capture
                            printing = False
                            #store the current Hirshfeld block to list: 'extracted_blocks'
                            extracted_blocks.append(current_hirsh_block)
                            continue #continue the loop for the next potential block

                        else: #if we haven't hit end of the/a Hirsh block, keep capturing lines until we do
                            current_hirsh_block.append(stripped_line)
                            
        except Exception as e:
            print(f"Could not read .out {orca_dft_output_filepath.name}: {e}")

    #get the last Hirshfeld block from an .out
    last_hirsh_block = extracted_blocks[-1]
    trimmed_hirsh_block = last_hirsh_block[6:-1]
                    
    # #we want just the important bits; no headers/trailing lines
    hirshfeld_block = []
    for line in trimmed_hirsh_block:
        stripped_line = line.strip()
        hirsh_line = stripped_line.split()
        hirshfeld_block.append(hirsh_line)

    #generate a df from the formatted text
    hirshfeld_df = pd.DataFrame(hirshfeld_block, columns=["atom_idx", "atom_symbol", "charge", "spin"])
    hirshfeld_df['atom_idx'] = hirshfeld_df['atom_idx'].astype(int)

    #round relevant cols to 4dps (used throughout proj)
    hirshfeld_df['charge'] = hirshfeld_df['charge'].astype(float).round(decimals=4)
    hirshfeld_df['spin'] = hirshfeld_df['spin'].astype(float).round(decimals=4)

    #return the DF object
    return hirshfeld_df

In [ ]:
def get_aryne_hirshfeld_charges(path_to_dft_out_files: Path, working_df: pd.DataFrame):
    """
    Function:
        Extract aryne atom Hirshfeld charges from Orca.out files
        For each output file:
        - Get aryne bond atom indices from the corresponding pd.DataFrame
        - Extract Hirshfeld text block from Orca.out
        - Parse Hirshfeld block for the charges corresponding to the two aryne C atoms
        
    Input:
        - path_to_dft_out_files: Path; A path to directory containing DFT output files from ORCA (must have !Hirshfeld in rt.)
        - working_df: pd.DataFrame that contains aryne atom indices in columns 'Aryne_Aindx_1' and 'Aryne_Aindx_2'
        
    Returns:
        - hirsh_chgs_Aindx_1: list; 
        - hirsh_chgs_Aindx_2: list; 
    """
    
    #get the list of .sdf files in /path_to_sdf_files/ to extract from
    target_out_files = get_filepaths_in_target_dir(path_to_dft_out_files, ".out", printing=False)
    print(f"Extracting Aryne Hirshfeld Charges from {len(target_out_files)} Orca.outs in '/{path_to_dft_out_files.name}/'...") #debug

    #Will return these two lists of charges
    hirsh_chgs_Aindx_1 = []
    hirsh_chgs_Aindx_2 = []
    
    #loop over all the target output files 1 at a time
    for out_file in target_out_files:
        
        #get the aryne_id from the filename
        file_name = out_file.stem
        file_name_parts = str(file_name).split('_')
        ar_id = f'{file_name_parts[0]}_{file_name_parts[1]}'
        print(f'Current File: {out_file.name}') #debug

        ### extract the Hirshfeld text section from .out as a pd.df
        hirsh_df = get_hirshfeld_textblock_orca(out_file)

        ### Now extract the correct aryne indices from the param_df
        #set up a condition to match our target aryne_id
        condition_mask = working_df['aryne_ID'] == ar_id
        
        #grab the aryne atom indices in the row matching our specified aryne
        selected_row = working_df.loc[condition_mask, ['Aryne_Aindx_1', 'Aryne_Aindx_2']]
        #these are 0-indexed atom-indices corresponding to Carbons in the aryne bond
        aryne_idx1 = selected_row['Aryne_Aindx_1'].values[0].astype(int)
        aryne_idx2 = selected_row['Aryne_Aindx_2'].values[0].astype(int)
        # print(f'aryne_idx1 = {aryne_idx1}') #debug
        # print(f'aryne_idx2 = {aryne_idx2}') #debug

        ### Now get the matching charges from our Hirshfeld DFs
        chg1_condition = hirsh_df['atom_idx'] == aryne_idx1
        matched_lines = hirsh_df.loc[chg1_condition]
        ary_chg1 = matched_lines.iat[0, 2]
        #add to the list
        hirsh_chgs_Aindx_1.append(ary_chg1)
        
        chg2_condition = hirsh_df['atom_idx'] == aryne_idx2
        matched_lines = hirsh_df.loc[chg2_condition]
        ary_chg2 = matched_lines.iat[0, 2]
        hirsh_chgs_Aindx_2.append(ary_chg2)
        
        # print(f'Hirshfeld Charge at atom_idx {aryne_idx1} = {ary_chg1}')
        # print(f'Hirshfeld Charge at atom_idx {aryne_idx2} = {ary_chg2}')

    #return the hirshfeld charges as lists corresponding to either atom index
    return hirsh_chgs_Aindx_1, hirsh_chgs_Aindx_2

In [ ]:
def add_abs_diff_bond_angles(working_dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Function:

    Input:
        - working_dataframe: pd.DataFrame

    Returns:
        - appended_dataframe: pd.DataFrame
    """

    # Check if required columns exist
    bond_angle_cols = ['Aidx1_BndAngl', 'Aidx2_BndAngl']

    #check the incoming DF for the required data
    for col in bond_angle_cols:
        #error handling in case col isn't found
        if col not in working_dataframe.columns:
            raise ValueError(f"Column '{col}' not found in passed pd.DataFrame")

    #init a list to store the calculated abs dif.
    absolute_differences = []

    #go through DF row-by-row, calculating abs dif and appending float to list
    for index, row in working_df.iterrows():
        try: #try/except handling for robustness
            #capture the first/second bond angle measurements (degrees)
            bond_angle1 = float(row['Aidx1_BndAngl']) #explicitly float just in case
            bond_angle2 = float(row['Aidx2_BndAngl'])

            #calculate the dif.
            abs_dif = abs(bond_angle1 - bond_angle2)

            #append the dif to our rolling list of calc'd vals:
            absolute_differences.append(abs_dif)

        except (ValueError, TypeError):
            #if can't be converted/calculated, append a NaN for that struc.
            absolute_differences.append(np.nan)

    #append the new data to the DataFrame
    working_dataframe['Abs_Dif_Bnd_Angl'] = absolute_differences

    return working_dataframe

In [ ]:
""" ### Determine Ring Size of Ring with Aryne Bond ###

This cell calculates the size of the ring (number of atoms comprising the ring) that includes the aryne bond.
An integer value representing the size of the aryne ring will be appended to the working DF. 
"""
#init a copy of the incoming DF
working_df = incoming_data.copy()

### Calculate the size of the ring that the aryne is in ###
size_of_aryne_ring = ring_size_of_aryne(aryne_opt_sdf_path)

#append aryne ring size data to working df
working_df['Sz_Aryne_Ring'] = size_of_aryne_ring

#debug
working_df.head()

In [ ]:
""" ### Determine whether the Aryne is Fused or Mono-cyclic ###

This cell will evaluate the ring system that the aryne is in to determine if the ring is monocyclic
or is fused to another ring (multi-cyclic). The result is appended to working_df. 
"""

### Determine if aryne is monocyclic or not; append to df as 'mono/multi' ###
mono_vs_multi_ring_data = is_aryne_in_fused_ring(aryne_opt_sdf_path)

#append mono vs. fused aryne rings data to working df
working_df['RingSys_Type'] = mono_vs_multi_ring_data

#debug
working_df.head()

In [ ]:
""" ### Determine Aryne C#C Bond's Atom Indices ### 

This cell is used to retain the atom indices of the atoms comprising the C#C triple bond.
*Note: Atom indices are 0-indexed, not 1-indexed. Atom indices are appended to the Pandas
DataFrame working_df. 
"""

### Get the aryne bond indices and append them to the DF (0-indexed) ###
target_sd_files = get_filepaths_in_target_dir(aryne_opt_sdf_path, ".sdf", printing=False)

#output some text
print(f"Determining C#C Atom Indices comprising the Aryne Bond for {len(target_sd_files)} .sdfs located in '/{aryne_opt_sdf_path.name}/...\n")

#list to store the captured C#C bond indices
aryne_bond_indices = []

#Loop over the sdfs in the target directory
for sd_file in target_sd_files:
    #print(f"Working file: {sd_file.name}")
    
    #use the get_aryne_bond_indices method to get a aryne indices as [x, y]
    bond_indices = get_aryne_bond_indices(sd_file)

    #store the captured indices to a list for unpacking in a moment
    aryne_bond_indices.append(bond_indices)

#separate the list of lists (unzipping to distinct lists)
aryne_atom_1_list = [item[0] for item in aryne_bond_indices]
aryne_atom_2_list = [item[1] for item in aryne_bond_indices]

#append each list of atom indices to the DF
working_df['Aryne_Aindx_1'] = aryne_atom_1_list
working_df['Aryne_Aindx_2'] = aryne_atom_2_list

#output some text
print(f"Finished determining aryne bond atom indices for {len(target_sd_files)} .sdfs located in '/{aryne_opt_sdf_path.name}/...\n")

#debug
working_df.head()

In [ ]:
""" ### Determine Aryne atom types (Not very relevant in this proj. but generally useful) ### 

This cell will append the atom types (ex: Carbon => C, Oxygen => O) to the DF. In this case, all
arynes are C#C bonds, so it's a bit redundant but may be applicable to future projects. The atom
types are appended to working_df. 
"""

#pull 0-indexed atom_ids from our working_df
ary_idx1_list = working_df['Aryne_Aindx_1'].tolist()
ary_idx2_list = working_df['Aryne_Aindx_2'].tolist()

# Get a list of targeted .sdfs in a dir
target_sd_files = get_filepaths_in_target_dir(aryne_opt_sdf_path, ".sdf", printing=False)

#output some text
print(f"Determining C#C Atom Types comprising the Aryne Bond for {len(target_sd_files)} .sdfs located in '/{aryne_opt_sdf_path.name}/...\n")

#list to store atom types
idx1_atom_types = []
#work through the target files
for sd_file, atom_index in zip(target_sd_files, ary_idx1_list):
    
    #convert to mol
    mol = sdf_to_mol_rdkit(sd_file)
    
    #find the atom based on the passed atom index
    atom = mol.GetAtomWithIdx(atom_index)
    atom_type = atom.GetSymbol()
    idx1_atom_types.append(atom_type)

idx2_atom_types = []
#work through the target files
for sd_file, atom_index in zip(target_sd_files, ary_idx2_list):
    #print the current file 
    #print(f"Current File: {sd_file.name}")
    
    #convert to mol
    mol = sdf_to_mol_rdkit(sd_file)
    
    #find the atom type based on the passed atom index
    atom = mol.GetAtomWithIdx(atom_index)
    atom_type = atom.GetSymbol()
    idx2_atom_types.append(atom_type)

#append the atom types to working_df
working_df['Aindx_1_Type'] = idx1_atom_types
working_df['Aindx_2_Type'] = idx2_atom_types

#output some text
print(f"Finished determining aryne bond atom types for {len(target_sd_files)} .sdfs located in '/{aryne_opt_sdf_path.name}/...\n")

#debug
working_df.head()

In [ ]:
""" ### Calculate Aryne Bond Distances (Angstroms) ###

This cell is used to calculate aryne C#C triple bond distances in units of Angstroms. 
The cell relies on the "calc_all_aryne_bond_distances" method to extract distances based
on atom indices from validated .sdfs. Values are appended to working_df. 
"""
# Get a list of targeted .sdfs in a dir
target_sd_files = get_filepaths_in_target_dir(aryne_opt_sdf_path, ".sdf", printing=False)

#output some text
print(f"Calculating Aryne bond distances for {len(target_sd_files)} .sdfs located in '/{aryne_opt_sdf_path.name}/...\n")

### Calculate the aryne bond distances in Angstroms and append to working_df ###
aryne_bond_distances = calc_all_aryne_bond_distances(aryne_opt_sdf_path, printing=False)

#append the calculated bond distances to the DF
working_df['Aryne_Bond_Dist(A)'] = aryne_bond_distances

#output some text
print(f"Finished calculating Aryne bond distances for {len(target_sd_files)} .sdfs located in '/{aryne_opt_sdf_path.name}/...\n")

#debug
working_df.head()

In [ ]:
""" ### Calculate Aryne Bond Angles, Indices, and Atom Types ### 

This cell first retains two sets of atom indices for 2 distinct aryne bond angles; one set of 3 atom
indices for the first aryne bond angle (ex: [1, 2, 3] for a bond angle between atom indices 1, 2 and 3)
and a second trio of atom indices for the other aryne bond angle. The values are appended to working_df. 
"""

#pull 0-indexed atom_ids from our working_df - these are the aryne bond indices
#pass these atom indices to have a direct reference to the aryne 
ary_idx1_list = working_df['Aryne_Aindx_1'].tolist()
ary_idx2_list = working_df['Aryne_Aindx_2'].tolist()

#output some text
print(f"Determining Aryne Bond Angle indices...")

#obtain the angle indices for each of the internal angles around the aryne C#C
idx1_angle_indices = get_aryne_bond_angle_indices(aryne_opt_sdf_path, ary_idx1_list)
idx2_angle_indices = get_aryne_bond_angle_indices(aryne_opt_sdf_path, ary_idx2_list)

#Next obtain the atom types comprising the aryne angle indices
target_sd_files = get_filepaths_in_target_dir(aryne_opt_sdf_path, ".sdf", printing=False)

#output some text
print(f"\nDetermining Atom Types for atom indices comprising aryne internal angles...")

#Get the atom types for the first bond angles
ang1_atom_types = []
#work through the target files
for sd_file, bond_index_set in zip(target_sd_files, idx1_angle_indices):
    #convert sdf to mol
    mol = sdf_to_mol_rdkit(sd_file)
    
    #init/reset a list to store the atom types corresponding to the atom indices 
    current_sdf_types = []

    #the bond_index_set is a list of atom indices, ex: [0, 1, 2]; iterate
    #over each atom index and get its atom type
    for atom_idx in bond_index_set:
        #find the atom based on the passed atom index
        atom = mol.GetAtomWithIdx(atom_idx)
        #get the atomic symbol corresponding with the index
        atom_type = atom.GetSymbol()
        #append the atom type to a working list for the current mol
        current_sdf_types.append(atom_type)
    #append the constructed list (a trio of atom types) to a list
    ang1_atom_types.append(current_sdf_types)

#Get the atom types for second bond angles
ang2_atom_types = []
#work through the target files
for sd_file, bond_index_set in zip(target_sd_files, idx2_angle_indices):
    #convert sdf to mol
    mol = sdf_to_mol_rdkit(sd_file)
    
    #init/reset a list to store the atom types corresponding to the atom indices 
    current_sdf_types = []

    #the bond_index_set is a list of atom indices, ex: [0, 1, 2]; iterate
    #over each atom index and get its atom type
    for atom_idx in bond_index_set:
        #find the atom based on the passed atom index
        atom = mol.GetAtomWithIdx(atom_idx)
        #get the atomic symbol corresponding with the index
        atom_type = atom.GetSymbol()
        #append the atom type to a working list for the current mol
        current_sdf_types.append(atom_type)
    #append the constructed list (a trio of atom types) to a list
    ang2_atom_types.append(current_sdf_types)

#output some text
print(f"\nCalculating Aryne Internal Bond Angles...")

### Calculate internal bond angles around the aryne atoms; add to DF ### 
idx1_bond_angles = calculate_bond_angles(aryne_opt_sdf_path, idx1_angle_indices)
idx2_bond_angles = calculate_bond_angles(aryne_opt_sdf_path, idx2_angle_indices)

#Append the data to the working DF
working_df['Aidx1_BndAngl_idxs'] = idx1_angle_indices #angle indices [0, 1, 2]
working_df['BndAng1_Atom_Types'] = ang1_atom_types    #angle atom types [X, C, C]
working_df['Aidx1_BndAngl'] = idx1_bond_angles        #calculated bond angles (floats)

#Also append data for the second angles
working_df['Aidx2_BndAngl_idxs'] = idx2_angle_indices #angle indices [0, 1, 2]
working_df['BndAng2_Atom_Types'] = ang2_atom_types    #angle atom types [X, C, C]
working_df['Aidx2_BndAngl'] = idx2_bond_angles        #calculated bond angles (floats)

#output some text
print(f"\nFinished calculating bond angle data.")

#debug
working_df.head()

In [ ]:
""" ### Calculate the Maximum Internal Angle around the Aryne C#C bond ###

This cell compares the internal bond angles on either side of the aryne C#C and 
retains the maximum internal angle (the larger one). The values are appended to 
working_df. 
"""

### Determine the maximum internal angle based on the two measured angles; add to DF ### 
working_df['Max_Aryne_BndAngl'] = working_df[['Aidx1_BndAngl', 'Aidx2_BndAngl']].max(axis=1)
working_df.head()

In [ ]:
""" ### Calculate the Max Deviation of Aryne Bond Angle from Ideal and Sum of Abs. Deviation ###

This cell is used to calculate the deviation of a given angle from its ideal bond angle based on the formula
for internal angles of equilateral polygons. The maximum deviation is returned, indicating that a particular 
bond angle is significantly strained. Similarly, the cell will calculate the Sum of the Absolute deviation, 
the sum of both angles' deviation from ideal. The values are appended to the working_df. 
""" 

### Adding to DF the deviation in optimized internal bond angles from "ideal" polygons ### 

#initialize some new columns as placeholders for new data
working_df['Aidx1_BndAngl_dev'] = working_df['Aidx1_BndAngl']
working_df['Aidx2_BndAngl_dev'] = working_df['Aidx2_BndAngl']

#calc the ideal internal angles based on size of ring; 5 is 108 deg, 6 is 120 deg.
ideal_five = (((5 - 2) * 180) / 5)
ideal_six = (((6 - 2) * 180) / 6)

#calculate deviation for the first bond angle index
working_df.loc[working_df['Sz_Aryne_Ring'] == 5, 'Aidx1_BndAngl_dev'] = abs(working_df['Aidx1_BndAngl'] - ideal_five)
working_df.loc[working_df['Sz_Aryne_Ring'] == 6, 'Aidx1_BndAngl_dev'] = abs(working_df['Aidx1_BndAngl'] - ideal_six)

#calculate deviation for the second bond angle index
working_df.loc[working_df['Sz_Aryne_Ring'] == 5, 'Aidx2_BndAngl_dev'] = abs(working_df['Aidx2_BndAngl'] - ideal_five)
working_df.loc[working_df['Sz_Aryne_Ring'] == 6, 'Aidx2_BndAngl_dev'] = abs(working_df['Aidx2_BndAngl'] - ideal_six)

### determine which of the two aryne internal angles is "more deviated from the ideal"
working_df['Max_angle_dev'] = working_df[['Aidx1_BndAngl_dev', 'Aidx2_BndAngl_dev']].max(axis=1)

#also try the sum of absolute deviation
working_df['Sum_abs_dev'] = working_df['Aidx1_BndAngl_dev'] + working_df['Aidx2_BndAngl_dev']

working_df.head()

In [ ]:
""" ### Calculate Aryne Dihedral Angles ###

This cell will calculate the dihedral angle (degrees) of the dihedral centered on the aryne bond indices.
The data are then appended to the working_df.
"""

# Extract dihedral angle data from optimized .sd-files
aryne_dihedral_angles = calc_all_aryne_dihedral_angles(aryne_opt_sdf_path)

#append dihedral angle data to working df
working_df['Aryne_DiH_Ang'] = aryne_dihedral_angles

#debug
working_df.head()

In [ ]:
""" ### extract HOMO/LUMO data from Orca .out files ### 

This cell is used to extract the HOMO and LUMO (FMO) Energies in electron volts (eV) from a validated
DFT .out file (Orca 5.0.3). The energies are appended to a working_df. 
"""

### extract HOMO/LUMO data from Orca .out files ### 
homo_energies, lumo_energies = extract_homo_lumo_energy_data(aryne_dft_out_path)

#append to working_df
working_df['eHOMO(eV)'] = homo_energies
working_df['eLUMO(eV)'] = lumo_energies

#also add the HOMO/LUMO Gap to the df (in eV)
working_df['H_L_Gap(eV)'] = pd.to_numeric(working_df['eLUMO(eV)']) - pd.to_numeric(working_df['eHOMO(eV)'])

#debug
working_df.head()

In [ ]:
#Extract Hirshfeld Charges from the aryne atoms
#Pass a (directory_containing_outs, working_PD_dataframe) 
hirsh_charges_aidx1, hirsh_charges_aidx2 = get_aryne_hirshfeld_charges(aryne_dft_out_path, working_df)

# #Append the charge data at each aryne atom to working_DF
working_df['Hirsh_chgs_Aindx_1'] = hirsh_charges_aidx1
working_df['Hirsh_chgs_Aindx_2'] = hirsh_charges_aidx2

#also try the sum of the absolute charge across the aryne; closer to 2 => more chg separation across aryne
working_df['Sum_Abs_Hirsh_aryne'] = abs(working_df['Hirsh_chgs_Aindx_1']) + abs(working_df['Hirsh_chgs_Aindx_2'])

#debug
working_df.head()

In [ ]:
#Calculate the absolute difference in bond angles between aryne bond angles (degrees)
add_abs_diff_bond_angles(working_df)

In [ ]:
#output the current working parameter DF as a .csv to local dir (mod7)
csv_name = 'Aryne_Calculated_Molecular_Descriptors'
working_df.to_csv(f'{csv_name}.csv', index=False)

#Also send a backup to /csv_backups
backup_dest = project_dir / "csv_backups" / "Calculated_DFT_descriptors" / "Aryne_Calculated_Molecular_Descriptors.csv"
working_df.to_csv(backup_dest, index=False)

In [ ]:
### Split DF into a 5-membered df ###
five_membered_df = working_df[working_df['Sz_Aryne_Ring'] == 5].copy()
print(f'5-membered arynes split into new df: {len(five_membered_df)}')

#output the current working parameter DF as a .csv to local dir (mod7)
csv_name = '5_membered_Calculated_Molecular_Descriptors'
five_membered_df.to_csv(f'{csv_name}.csv', index=False)

#Also send a backup to /csv_backups
backup_dest = project_dir / "csv_backups" / "Calculated_DFT_descriptors" / "5_membered_Calculated_Molecular_Descriptors.csv"
five_membered_df.to_csv(backup_dest, index=False)

In [ ]:
### Split DF into a 6-membered df ###
six_membered_df = working_df[working_df['Sz_Aryne_Ring'] == 6].copy()
print(f'6-membered arynes split into new df: {len(six_membered_df)}')

#output the current working parameter DF as a .csv to local dir (mod7)
csv_name = '6_membered_Calculated_Molecular_Descriptors'
six_membered_df.to_csv(f'{csv_name}.csv', index=False)

#Also send a backup to /csv_backups
backup_dest = project_dir / "csv_backups" / "Calculated_DFT_descriptors" / "6_membered_Calculated_Molecular_Descriptors.csv"
six_membered_df.to_csv(backup_dest, index=False)

In [ ]:
#Next step is to plot the data in univariate plots